In [3]:
import functools
from typing import List

import numpy as np
import jax
import jax.numpy as jnp

from jax.experimental.pallas.ops.tpu.megablox.gmm import gmm
from jax.experimental.shard_map import shard_map
from jax.sharding import Mesh, NamedSharding, PartitionSpec

In [4]:
P = PartitionSpec

mesh = Mesh(jax.devices(), ('x',))

seq_p_spec = P('x',)
seq_p_sharding = NamedSharding(mesh, seq_p_spec)

In [5]:
def print_sharded(x):
    if not isinstance(x, jax.Array):
        print("not a JAX arry")
        return
    print(f"x shape: {x.shape} x sharding: {x.sharding}")
    if not x.is_fully_replicated:
        for i, shard in enumerate(x.addressable_shards):
            print(f"shard {i}: {shard.data}")
    else:
        print(f"x is fully replicated: \n {x}")

In [6]:
dtype = jnp.bfloat16
N_TOKENS = 512 # number of tokens in the current batch
E = 16 # number of global experts
EP_SIZE = mesh.size
D = 1024 # n_head * head_dim
F = 2048 # intermediate size
TOP_K = 2

In [7]:
x = jax.random.normal(jax.random.PRNGKey(0), (N_TOKENS, D))
expert_gating_weight = jax.random.normal(jax.random.PRNGKey(1), (D, E))

x_sharding = P('x', None)

x = jax.device_put(x, NamedSharding(mesh, x_sharding))
expert_gating_weight = jax.device_put(expert_gating_weight, NamedSharding(mesh, P()))

# Expert selection (Routing)

The first component in MoE layer is the routing module. Which each `input_token` to `top_k` experts. Then in the FFN module, each token will be computed by the selected `top_k` experts.

Given:
`T`: num_tokens
`D`: model_dim (num_heads * head_dim)
`E`: number of experts


Given transformer output `x` with shape `[T, D]`, the router has a weight (`router_w`) of shape `[D, E]`. Then the topk is calculated as `top_k_logits, top_k_indices = topk(softmax(x @ router_w, dim=-1))`. `top_k_logits` is the softmax logtis on the topk i


When `top_k > 1`, the activation from multiple experts will be combined by taking the weighted sum with the `softmax logits` of the `topk` experts. ** Note that the weighted sum maybe calculated on the 1st gmm, or the 2nd gmm, depending on the model arch**

In [8]:
def sort_indices_topk(x, top_k_indices):
    """
    Utility function to argsort x indices according to top_k_indices

    Args:
    x: [num_tokens, ...]
    top_k_indices: [num_tokens, top_k]

    Return:
    sorted_token_indices: token indices in sorted order, x can be sorted by x[sorted_token_indices]
    topk_argsort_revert_indices: x can be unsort by x[sorted_token_indices][topk_argsort_revert_indices]
    
    """
    n_tokens = x.shape[0]
    top_k = top_k_indices.shape[-1]

    top_k_indices = top_k_indices.flatten()
    topk_argsort_indices = jnp.argsort(top_k_indices)
    topk_argsort_revert_indices = topk_argsort_indices.argsort()

    token_indices = jnp.arange(n_tokens).repeat(top_k)
    sorted_token_indices = token_indices[topk_argsort_indices]

    return sorted_token_indices, topk_argsort_revert_indices

functools.partial(jax.jit,
                  static_argnames=["n_experts",
                                   "top_k"])
def expert_gating(x,
                  expert_gating_weight,
                  n_experts,
                  top_k):
    @functools.partial(shard_map,
                       mesh=mesh,
                       in_specs=(x_sharding,
                                 P()),
                       out_specs=(x_sharding, P('x',), P('x',), P('x',)),
                       check_rep=False)
    def expert_gating_internal(x,
                               expert_gating_weight):
        gating_out = x @ expert_gating_weight
        gating_logits = jax.nn.softmax(gating_out, axis=-1)
        top_k_logits, top_k_indices = jax.lax.top_k(gating_logits, top_k)
        sorted_token_indices, topk_argsort_revert_indices = sort_indices_topk(x, top_k_indices)
        
        # top_k_indices = top_k_indices.flatten()
        # top_k_logits = top_k_logits.flatten()
    
        # topk_argsort_indices = jnp.argsort(top_k_indices)
        # topk_argsort_revert_indices = topk_argsort_indices.argsort()

        # n_tokens = x.shape[0]
        # token_indices = jnp.arange(n_tokens).repeat(top_k)
        # sorted_token_indices = token_indices[topk_argsort_indices]
    
        histo = jnp.bincount(top_k_indices.flatten(), length=n_experts)

        sorted_tokens = x[sorted_token_indices]

        return sorted_tokens, histo, top_k_logits, topk_argsort_revert_indices

    return expert_gating_internal(x, expert_gating_weight)

In [9]:
sorted_tokens, histo, top_k_logits, topk_argsort_revert_indices = expert_gating(x, expert_gating_weight, n_experts=E, top_k=TOP_K)

In [10]:
repeated_x = jnp.repeat(x, TOP_K, axis=0)

# Test unsort work
# To recover the order after GMM
@functools.partial(shard_map,
                       mesh=mesh,
                       in_specs=(P('x', None), P('x')),
                       out_specs=P('x', None),
                       check_rep=False)
def unsort(sorted_tokens, topk_argsort_revert_indices):
    return sorted_tokens[topk_argsort_revert_indices]

tokens_in_orig_order = unsort(sorted_tokens, topk_argsort_revert_indices)
assert jnp.array_equal(tokens_in_orig_order, repeated_x)

# Test expert routing works (TODO)

In [11]:
ep_ranks = jnp.arange(EP_SIZE)
ep_ranks = jax.device_put(ep_ranks, NamedSharding(mesh, P('x')))

@functools.partial(shard_map,
                       mesh=mesh,
                       in_specs=(P('x'),
                                 P('x'),
                                 P()),
                       out_specs=(P('x')),
                       check_rep=False)
def calc_ep_group_metadata(histo, ep_ranks, ep_group_mask):
    ep_group_size = E // EP_SIZE
    ep_idx = ep_ranks[0]
    
    ep_group_send_size = histo @ ep_group_mask
    return ep_group_send_size

# Can generate offline
def gen_ep_group_size_mask(n_experts, ep_size):
    mask = jnp.zeros((ep_size, n_experts), dtype=jnp.int32)
    ep_group_size = n_experts // ep_size
    for i in range(ep_size):
        mask = mask.at[i, i*ep_group_size : (i+1)*ep_group_size].set(1)
    return mask.transpose()

In [12]:
print_sharded(histo)

# Can generate offline
mask = gen_ep_group_size_mask(E, EP_SIZE)
print(f"check mask: {mask}")
# This is the send sizes of ra2a.
ep_group_sizes = calc_ep_group_metadata(histo, ep_ranks, mask)
print_sharded(ep_group_sizes)

x shape: (128,) x sharding: NamedSharding(mesh=Mesh('x': 8, axis_types=(Auto,)), spec=PartitionSpec('x',), memory_kind=device)
shard 0: [ 7  8 15  5  4  6  7 10 11  7  8  7  7  7 10  9]
shard 1: [ 3  8 10 13  7  8 13  5  7  5  7 11 11  9  5  6]
shard 2: [ 3  6  5 13  9  9 12  9  3 10  8  4  8  7 12 10]
shard 3: [ 3  4  7  8 11 14 11  8  9 10  6  5 10 12  3  7]
shard 4: [10 13  6 11  7  4  7 10  4  7 10  4  8 14  8  5]
shard 5: [ 9  8  3  6 12 11  5  7  6  9 10 11 12  6  7  6]
shard 6: [10  9  7 11  7  7  8  6  7  8  7  7 10  9  5 10]
shard 7: [ 3 14 11  7  7  2  5  9  8 12  6  9  8 11  9  7]
check mask: [[1 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0]
 [0 0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0 0]
 [0 0 0 1 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [0 0 0 0 1 0 0 0]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 1]]
x shape: (64,) x sharding: NamedSharding(mesh=Mesh('x': 8, axis_types=(Auto,)), spec